In [ ]:
%config InlineBackend.figure_format = 'retina'

import ast
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns
import zstandard as zstd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

import difflib
import numpy as np
import scipy.stats as stats

from sklearn.base import clone
from sklearn.experimental import enable_halving_search_cv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, HalvingRandomSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier

COLOR_SUCCESS = "#009E73"
COLOR_FAIL = "#D55E00"

In [ ]:
# Load profiling and logprob data
def load_data(dataset, model_size):
    base_path = f"../logs/{dataset}/llamacpp_qwen3_{model_size}b"
    max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
    regex = r"logprob=(-?\d+\.\d+(?:[eE][-+]?\d+)?)"
    token_regex = r"ChatCompletionTokenLogprob\(token=(['\"])(.*?)\1, bytes="
    all_logprobs = []
    all_tokens = []
    dctx = zstd.ZstdDecompressor()
    for i in range(max_folder + 1):
        with open(f"{base_path}/{i}/run_0/raw/trace.json.zst", "rb") as f:
            compressed = f.read()
            data = dctx.decompress(compressed).decode("utf-8")
            trace = json.loads(data)
            trace = [t for t in trace if t["name"] == "ActionStep" and "logprob=" in t["attributes"]["output.value"]]
            trace = sorted(trace, key=lambda t: t["start_time"])
            logprobs = []
            tokens = []
            for t in trace:
                v = json.loads(t["attributes"]["output.value"])
                logprobs.append([
                    float(re.search(regex, l).group(1)) for l in v["model_output_message"]["raw"]["logprobs"]
                ])
                tokens.append([
                    re.search(token_regex, l).group(2) for l in v["model_output_message"]["raw"]["logprobs"]
                ])
            all_logprobs.append(logprobs)
            all_tokens.append(tokens)

    input_path = f"../data/{dataset}/profile_results_llamacpp_qwen3_{model_size}b_judged.csv"
    input_df = pd.read_csv(input_path)
    is_correct = input_df["agent_output_eval"] == "CORRECT"
    input_df["agent_output_is_correct"] = is_correct
    if model_size == "30":
        llm_eval_file = f"../data/{dataset}/llm_{dataset}_results_qwen3_30b_2507_judged.csv"
        llm_eval_col = "qwen3:30b-a3b-instruct-2507-q4_K_M_eval"
    else:
        llm_eval_file = f"../data/{dataset}/llm_{dataset}_results{'' if dataset == 'frames' else '_qwen3_' + model_size + 'b'}_judged.csv"
        llm_eval_col = f"qwen3:{model_size}b_eval"
    llm_eval = pd.read_csv(llm_eval_file)
    is_llm_correct = llm_eval[llm_eval_col] == "CORRECT"

    # Only consider traces where the LLM doesn't already know the answer
    all_logprobs = [all_logprobs[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    all_tokens = [all_tokens[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    is_correct = [is_correct[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    valid_idx = [i for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    all_data = input_df.iloc[valid_idx].copy()

    max_steps = 10
    exit_energy = all_data["step_0_energy_total_mWh"].copy()
    for step in range(max_steps):
        all_data[f"exit_energy_step_{step}"] = exit_energy
        try:
            step_energy = all_data[f"step_{step + 1}_energy_total_mWh"].fillna(0)
        except:
            print(f"Step {step + 1} is not found.")
            break
        exit_energy += step_energy

    all_data["total_energy"] = exit_energy

    return (all_logprobs, all_tokens, is_correct, valid_idx, all_data)

def lcs(a: str, b: str) -> str:
    matcher = difflib.SequenceMatcher(None, a, b)
    match = matcher.find_longest_match(0, len(a), 0, len(b))
    if match.size == 0:
        return ""
    return a[match.a: match.a + match.size]

In [ ]:
dataset = "frames"
size = "30"
all_logprobs, all_tokens, is_correct, valid_idx, all_data = load_data(dataset, size)

In [ ]:
# Analyze success rate and energy usage

print(f"Task Accuracy: {round(np.mean(is_correct), 3)}")

max_steps = 10

exit_energy = all_data["step_0_energy_total_mWh"].copy()
for step in range(max_steps):
    all_data[f"exit_energy_step_{step}"] = exit_energy
    try:
        step_energy = all_data[f"step_{step + 1}_energy_total_mWh"].fillna(0)
    except:
        print(f"Step {step + 1} is not found.")
        break
    exit_energy += step_energy

all_data["total_energy"] = exit_energy

fails = all_data[~all_data["agent_output_is_correct"]]
successes = all_data[all_data["agent_output_is_correct"]]
for label, data in [("All", all_data), ("Fails", fails), ("Successes", successes)]:
    total_energy = data["total_energy"]
    mean_energy = total_energy.mean()
    sem = stats.sem(total_energy)  # standard error of the mean
    confidence = 0.95
    ci = stats.t.interval(confidence, df=len(total_energy)-1, loc=mean_energy, scale=sem)
    print(f"{label}: n {len(total_energy)} | mean {mean_energy:.1f} | median {total_energy.median():.1f} | std {total_energy.std():.1f} | sem {sem:.1f} | 95% CI [{ci[0]:.1f}, {ci[1]:.1f}]")

In [ ]:
print(all_data["duration_sec"].mean())

output_tokens = all_data["step_0_llm.token_count.completion"].copy()
for step in range(1, max_steps):
    output_tokens += all_data[f"step_{step}_llm.token_count.completion"].fillna(0)

print(output_tokens.mean())
print(output_tokens[~all_data["agent_output_is_correct"]].mean())
print(output_tokens[all_data["agent_output_is_correct"]].mean())

input_tokens = all_data[[f"step_{step}_llm.token_count.prompt" for step in range(max_steps)]].max(axis=1)

print(input_tokens.mean())
print(input_tokens[~all_data["agent_output_is_correct"]].mean())
print(input_tokens[all_data["agent_output_is_correct"]].mean())

In [ ]:
plt.figure(figsize=(6, 4))
ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.6)

for data, label, color in [(successes, "Success", COLOR_SUCCESS), (fails, "Fail", COLOR_FAIL)]:
    total_used_energy = data["total_energy"].sum()
    step_energy_contrib = []
    for step in range(max_steps):
        step_energy_contrib.append(data[f"exit_energy_step_{step}"].fillna(0).sum())

    step_energy_contrib_pct = [100 * energy / total_used_energy for energy in step_energy_contrib]
    step_energy_contrib_pct.append(100)
    plt.plot(np.arange(1, 12), step_energy_contrib_pct, label=label, color=color, marker="o")

plt.xticks(np.arange(1, 12))
plt.ylim(0, 105)

ax.tick_params(axis='both', labelsize=20)
# ax.legend(fontsize=20)

plt.tight_layout()
# plt.savefig(f"../figures/step_energy_contrib_{dataset}_llamacpp_qwen3_{size}b.pdf", bbox_inches="tight", pad_inches=0.0)
plt.show()

In [ ]:
# Display barplot of number of successes/failures by number of steps taken to finish the task
incorrect_steps = [0] * 11
correct_steps = [0] * 11
for i, logprobs in enumerate(all_logprobs):
    n = len(logprobs)
    if is_correct[i]:
        correct_steps[n - 1] += 1
    else:
        incorrect_steps[n - 1] += 1

plt.figure(figsize=(6, 4))
ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.6)
for i in range(1, 12):
    plt.bar(i - 0.2, correct_steps[i - 1], width=0.4, color=COLOR_SUCCESS, label="Success" if i == 1 else "", edgecolor="black", linewidth=0.5)
    plt.bar(i + 0.2, incorrect_steps[i - 1], width=0.4, color=COLOR_FAIL, label="Fail" if i == 1 else "", edgecolor="black", linewidth=0.5)
plt.xticks(list(range(1, 12)))

# plt.xlabel("Number of steps")
# plt.ylabel("Count")
# plt.yticks([0, 25, 50, 75, 100, 125])
ax.tick_params(axis='both', labelsize=20)
# ax.legend(fontsize=20)
plt.tight_layout()
# plt.savefig(f"../figures/step_dist_{dataset}_llamacpp_qwen3_{size}b.pdf", bbox_inches="tight", pad_inches=0.0)
plt.show()

In [ ]:
def plot_logprobs_stats_by_step(func):
    step_logprobs_data = [[] for _ in range(11 * 2)]
    for i, logprobs in enumerate(all_logprobs):
        for step, step_logprobs in enumerate(logprobs):
            step_logprobs_data[step * 2 + int(is_correct[i])].append(func(step_logprobs))

    plt.figure(figsize=(8, 5))
    positions = []
    colors = []
    for i in range(1, 12):
        positions.extend([i-0.15, i+0.15])
        colors.extend(["red", "green"])
    
    for i in range(len(step_logprobs_data)):
        bp = plt.boxplot(
            [step_logprobs_data[i]],
            positions=[positions[i]],
            patch_artist=True,
            widths=0.25
        )

        box = bp['boxes'][0]
        box.set_facecolor(colors[i])
        box.set_linewidth(1)  # thinner border
        # Customize median line
        median = bp['medians'][0]
        median.set_color('black')

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])
    xticks = np.arange(1, 12, 1)
    ax.set_xlim(0, 12)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{y}" for y in xticks])
    
    plt.xlabel("Step")
    func_name = func.__name__.split(".")[-1]
    func_name = func_name[0].upper() + func_name[1:]
    plt.ylabel(f"{func_name} logprobs")
    plt.title(f"{func_name} logprobs vs Step")
    plt.show()

In [ ]:
plot_logprobs_stats_by_step(min)

In [ ]:
plot_logprobs_stats_by_step(np.mean)

In [ ]:
def plot_step_logprobs_line(func, diff=False):
    plt.figure(figsize=(8, 5))

    for i, logprobs in enumerate(all_logprobs):
        line = np.array([func(step_logprobs) for step_logprobs in logprobs])
        if diff:
            line = line[1:] - line[:-1]
        color = "green" if is_correct[i] else "red"
        style = "dotted" if is_correct[i] else "dotted"
        alpha = 0.8 if is_correct[i] else 0.8
        if len(line) == 1:
            plt.scatter([-0.1], line, color=color, s=20, zorder=3, alpha=alpha)
        else:
            plt.plot(range(len(line)), line, color=color, linestyle="--", alpha=0.6)

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])
    xticks = np.arange(0, 11, 1)
    ax.set_xlim(-1, 11)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{y}" for y in xticks])

    plt.xlabel("Step")
    func_name = func.__name__.split(".")[-1]
    func_name = func_name[0].upper() + func_name[1:]
    plt.ylabel(f"{func_name} logprobs{' change' if diff else ''}")
    plt.title(f"{func_name} logprobs{' change' if diff else ''} vs Step")

    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

def exp_saturation(x, L, k, c):
    return L * (1 - np.exp(-k * x)) + c

def fit_exp_saturation_curve(y):
    n = len(y)
    x = np.linspace(1, n, n)
    p0 = [max(y), 1.0, 0.0]
    params, _ = curve_fit(
        exp_saturation,
        x,
        y,
        p0=p0,
        bounds=([0.0, -np.inf, -np.inf], [1.0, np.inf, np.inf]),
        maxfev=10000,
    )
    return params

def plot_exp_saturation_curve(n, params, x_offset=0, **kwargs):
    L, k, c = params
    x = np.linspace(1, n, 100)
    y = exp_saturation(x, L, k, c)
    plt.plot(x_offset + x, y, **kwargs)

def plot_min_logprobs(
    start_step,
    end_step,
    num_logprob=10,
    exp=False,
    fit=False,
    normalize=False,
    trace_type=None,
    save=None,
):
    assert start_step <= end_step, "Start step must not be greater than end step"
    plt.figure(figsize=(16, 4))
    cnt = 0

    step_count = {}
    step_success_count = {}
    for i, logprobs in enumerate(all_logprobs):
        n = len(logprobs)
        step_count[n] = step_count.get(n, 0) + 1
        if is_correct[i]:
            step_success_count[n] = step_success_count.get(n, 0) + 1

    max_steps = max(step_count.keys())
    for num_step in range(max_steps, 0, -1):
        step_count[num_step] = step_count.get(num_step, 0) + step_count.get(num_step + 1, 0)
        step_success_count[num_step] = step_success_count.get(num_step, 0) + step_success_count.get(num_step + 1, 0)

    for i, logprobs in enumerate(all_logprobs):
        # if len(logprobs) <= end_step:
        #     continue
        if trace_type is not None:
            if is_correct[i] != trace_type:
                continue
        cnt += 1
        # line = np.array([np.sort(step_logprobs)[:num_logprob] for step_logprobs in logprobs[:max_step]]).flatten()
        color = COLOR_SUCCESS if is_correct[i] else COLOR_FAIL
        # color = "#542788" if is_correct[i] else "#2D862D"
        # style = "-" if is_correct[i] else "dotted"
        style = "-" if not fit else "dotted"
        # alpha = 0.8 if is_correct[i] else 0.4
        # plt.scatter(range(len(line)), line, color=color, alpha=alpha)

        global_mean = np.mean([np.exp(v) for lp in logprobs[:end_step] for v in lp])
        global_std = np.std([np.exp(v) for lp in logprobs[:end_step] for v in lp])

        for step, step_logprobs in enumerate(logprobs[start_step-1:end_step]):
            if exp:
                step_logprobs = np.exp(step_logprobs)

            line = np.sort(step_logprobs)[:num_logprob]
            # line = np.quantile(step_logprobs, [0.0 + 0.05 * i for i in range(20)])

            if normalize:
                non_zero_step_logprobs = step_logprobs[step_logprobs < 1]
                # line = (line - np.mean(non_zero_step_logprobs)) / np.std(non_zero_step_logprobs)
                line = (line - global_mean) / global_std
                # line = (line - np.mean(line)) / np.std(line)

            alpha_correct = 1 - step_success_count[start_step + step] / step_count[start_step + step]
            alpha = alpha_correct if is_correct[i] else 1 - alpha_correct
            alpha = max(alpha, 0.3)

            if fit:
                params = fit_exp_saturation_curve(line)
                plot_exp_saturation_curve(num_logprob, params, x_offset=step * num_logprob, color=color, alpha=alpha, linestyle="-")

            start = step * num_logprob
            plt.plot(range(start, start + len(line)), line, color=color, alpha=alpha, linestyle=style)

    ax = plt.gca()
    num_steps = end_step - start_step + 1
    xticks = [(step * num_logprob + num_logprob // 2) for step in range(num_steps)]
    xlabels = [f"{step + 1}" for step in range(num_steps)]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels)
    ax.set_xlim((-1, num_steps * num_logprob))

    ax.tick_params(axis='both', labelsize=24)

    # plt.xlabel("Step")
    # plt.ylabel("Logprobs" if not exp else "Probabilities")
    # plt.title(f"Top {num_logprob} smallest token {'logprobs' if not exp else 'probabilities'} vs Step (n = {cnt})")

    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()

    # custom_legend = [
    #     Line2D([0], [0], color=COLOR_SUCCESS, lw=2, label="Success"),
    #     Line2D([0], [0], color=COLOR_FAIL, lw=2, label="Fail"),
    # ]
    # plt.legend(handles=custom_legend, fontsize=24, labelspacing=0.1, frameon=False, bbox_to_anchor=(1.0, -0.05), loc="lower right")

    if save:
        plt.savefig(save, bbox_inches="tight", pad_inches=0.0)
    plt.show()

def plot_logprob_evolution(idx, start_step, end_step, num_logprobs=10):
    assert start_step <= end_step
    logprobs = all_logprobs[idx]

    data = np.vstack([np.sort(lp)[:num_logprobs] for lp in logprobs[start_step-1:end_step]])
    num_steps, length = data.shape

    # Axes:
    # t = time index inside each time series
    # s = external time step (one series per step)
    t = np.arange(length)
    steps = np.arange(num_steps) + 1

    T, S = np.meshgrid(t, steps)

    # ---- Plot 3D surface ----
    colors = plt.cm.tab20(np.linspace(0, 1, length))
    # colors = plt.cm.viridis(np.linspace(0, 1, length))
    facecolors = np.repeat(colors[np.newaxis, :, :], num_steps, axis=0)

    plt.close('all')
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    # ax.plot_surface(T, S, data, facecolors=facecolors, shade=False)

    for j in range(length):
        ax.plot(
            np.full(num_steps, t[j]),  # x-axis → time index (constant)
            steps, # y-axis → time step
            data[:, j],          # z-axis → actual values
            color=colors[j],
            linewidth=1
        )

        ax.scatter(
            np.full(num_steps, t[j]),
            steps,
            data[:, j],
            color=colors[j],
            s=80,  # size of the spheres (larger = bigger)
            marker='o',
            edgecolor='k',  # optional: add black edge for better visibility
            alpha=0.9
        )

    for i in range(num_steps):
        ax.plot(
            t,                   # all logprob indices (x-axis)
            np.full(length, steps[i]),# constant step (y-axis)
            data[i, :],           # all logprob values for step i (z-axis)
            color='gray',
            linestyle='dashed',
            linewidth=1
        )

    ax.set_xlabel("Logprob index")
    ax.set_xticks(t)
    ax.set_ylabel("Step")
    ax.set_yticks(steps)
    ax.set_zlabel("Logprob")
    ax.set_title(f"Logprob evolution for trace {idx} ({'success' if is_correct[idx] else 'fail'})")

    ax.view_init(elev=30, azim=-135)
    plt.show()


In [ ]:
plot_min_logprobs(1, 10, num_logprob=10, exp=False, fit=False, normalize=False, trace_type=None)
# plot_min_logprobs(1, 10, num_logprob=10, exp=False, fit=False, normalize=False, trace_type=None, save=f"../figures/logprobs_dist_{dataset}_qwen3_{size}b.pdf")

In [ ]:
# %matplotlib widget
plot_logprob_evolution(15, 1, 11)

In [ ]:
import difflib
import numpy as np
import scipy.stats as stats

from sklearn.base import clone
from sklearn.experimental import enable_halving_search_cv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, HalvingRandomSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier

fpr_targets = [0.05 * i for i in range(21)]

def lcs(a: str, b: str) -> str:
    matcher = difflib.SequenceMatcher(None, a, b)
    match = matcher.find_longest_match(0, len(a), 0, len(b))
    if match.size == 0:
        return ""
    return a[match.a: match.a + match.size]

def normalize(arr):
    return (arr - np.mean(arr)) / np.std(arr)

# ------------------ 1) Prepare training data ------------------ #
def extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=10):
    X_features = []
    y = []
    valid = []

    for i, (logprobs, tokens) in enumerate(zip(logprobs_data, tokens_data)):
        if len(logprobs) <= clf_step:
            continue
        feature_vector = []

        probs = [np.exp(lp) for lp in logprobs[:clf_step]]
        min_logprobs = [v for p in probs for v in np.sort(p)[:num_logprobs]]
        feature_vector.extend(min_logprobs)

        # Token features
        num_tokens = [len(lp) for lp in logprobs[:clf_step]]
        feature_vector.extend(num_tokens)

        num_thought_tokens = [(step_tokens.index("<code") if "<code" in step_tokens else len(step_tokens)) for step_tokens in tokens[:clf_step]]
        feature_vector.extend(num_thought_tokens)

        # Length of longest common substring between current step and previous step
        for j in range(max(clf_step - 1, 1), clf_step):
            cur_gen = "".join(tokens[j])
            prev_gen = "".join(tokens[j - 1])
            feature_vector.append(len(lcs(cur_gen, prev_gen)) / len(cur_gen))

        X_features.append(feature_vector)
        y.append(int(labels[i]))
        valid.append(i)

    if all(y) or not any(y):
        print("Not enough valid labels")
        raise Exception

    max_num_feature = max(len(f) for f in X_features)
    correct_record_idx = [i for i in range(len(X_features)) if len(X_features[i]) == max_num_feature]
    valid = [valid_idx for i, valid_idx in enumerate(valid) if len(X_features[i]) == max_num_feature]

    X_features = [f for f in X_features if len(f) == max_num_feature]
    X = np.array(X_features)
    y = np.array(y)[correct_record_idx]
    
    return X, y, valid

def stats_at_fpr(y_true, y_proba, fpr_targets, clf_step, data):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    results = {}
    batch_energy_wastage = data[~data["agent_output_is_correct"]]["total_energy"].sum()
    num_pos = y_true.sum()

    for target in fpr_targets:
        valid = np.where(fpr <= target)[0]

        if len(valid) == 0:
            results[target] = {
                "tpr": 0.0,
                "thr": None,
                "energy_wastage": batch_energy_wastage,
                "energy_wastage_reduction": 0.0,
                "avg_energy_wastage": batch_energy_wastage / num_pos,
                "avg_energy_wastage_reduction": 0.0,
                "energy_wastage_reduction_pct": 0.0,
            }
        else:
            best_idx = valid[np.argmax(tpr[valid])]
            best_thr = thr[best_idx]

            preds = (y_proba >= best_thr).astype(np.bool)
            false_neg = (y_true & (~preds)).astype(np.bool)

            energy_wastage = data[preds][f"exit_energy_step_{clf_step - 1}"].sum() + data[false_neg]["total_energy"].sum()
            energy_wastage_reduction = batch_energy_wastage - energy_wastage

            results[target] = {
                "tpr": tpr[best_idx],
                "thr": best_thr,
                "num_false_pos": np.sum(preds & ~y_true),
                "energy_wastage": energy_wastage,
                "energy_wastage_reduction": energy_wastage_reduction,
                "avg_energy_wastage": energy_wastage / num_pos,
                "avg_energy_wastage_reduction": energy_wastage_reduction / num_pos,
                "energy_wastage_reduction_pct": 100 * energy_wastage_reduction / batch_energy_wastage,
            }

    return results

def train_classifier(
    logprobs_data,
    tokens_data,
    all_data,
    labels,
    clf_step,
    n_folds=5,
    num_logprobs=10,
):
    num_total_negative = len(labels) - sum(labels)
    X, y, valid = extract_features(
        logprobs_data,
        tokens_data,
        labels,
        clf_step,
        num_logprobs=num_logprobs,
    )

    num_total = len(y)
    num_positive = y.sum()
    num_negative = num_total - num_positive

    if min(num_positive, num_negative) <= n_folds:
        return
    
    total_energy_wastage = all_data[~all_data["agent_output_is_correct"]]["total_energy"].sum()
    valid_data = all_data.iloc[valid]
    
    print(f"X dimension: {X.shape}")
    print(f"y positive: {y.mean():.3f} ({num_positive}/{num_total})")

    base_estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    param_dist = {
        "max_depth": stats.randint(3, 8),
        "learning_rate": stats.loguniform(0.005, 0.2),
        "min_child_weight": stats.randint(1, 8),
        "subsample": stats.uniform(0.7, 0.3),
        "colsample_bytree": stats.uniform(0.7, 0.3),
        "gamma": stats.uniform(0, 5),

    }

    search = HalvingRandomSearchCV(
        estimator=base_estimator,
        param_distributions=param_dist,
        scoring="roc_auc",
        n_jobs=-1,
        cv=n_folds,
        factor=3,
        resource="n_estimators",
        max_resources=300,
        min_resources=10,
        random_state=42,
        verbose=0,
    )

    outer_cv = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=42,
    )

    oof_proba = np.zeros(len(y))

    fold_data = []

    print("\nRunning Nested CV...")

    for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X, y), 1):
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_val, y_val = X[va_idx], y[va_idx]

        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        # print("Best params:", search.best_params_)

        train_proba = best_model.predict_proba(X_train)[:, 1]
        val_proba = best_model.predict_proba(X_val)[:, 1]

        oof_proba[va_idx] = val_proba

        train_auc_score = roc_auc_score(y_train, train_proba)
        fold_auc_score = roc_auc_score(y_val, val_proba)
        
        data_val = valid_data.iloc[va_idx]
        stats_at_fpr_dict = stats_at_fpr(y_val, val_proba, fpr_targets, clf_step, data_val)

        fold_data.append({
            "fold": fold,
            "train_idx": tr_idx,
            "fold_idx": va_idx,
            "train_label": y_train,
            "fold_label": y_val,
            "train_prob": train_proba,
            "fold_prob": val_proba,
            "train_auc_score": train_auc_score,
            "fold_auc_score": fold_auc_score,
            "stats_at_fpr": stats_at_fpr_dict,
        })

    # -------------------------
    # Summary
    # -------------------------

    print("\n" + "=" * 50)
    print("FINAL NESTED CV RESULTS")
    print("=" * 50)

    train_aucs = [f["train_auc_score"] for f in fold_data]
    mean_train_auc = np.mean(train_aucs)
    std_train_auc = np.std(train_aucs)

    aucs = [f["fold_auc_score"] for f in fold_data]
    mean_fold_auc = np.mean(aucs)
    std_fold_auc = np.std(aucs)

    print(f"Train ROC-AUC: {mean_train_auc:.4f} ± {std_train_auc:.4f}")
    print(f"Test ROC-AUC: {mean_fold_auc:.4f} ± {std_fold_auc:.4f}")

    print("TPR and Energy Wastage @ FPR summary:")
    agg_fpr_stats = []
    t_crit = stats.t.ppf((1 + 0.95) / 2, df=n_folds-1)
    for fpr in fpr_targets:
        tprs = [f["stats_at_fpr"][fpr]["tpr"] for f in fold_data]
        mean_tpr = np.mean(tprs)
        std_tpr = np.std(tprs)

        num_false_pos = [f["stats_at_fpr"][fpr]["num_false_pos"] for f in fold_data]
        mean_overall_fpr = np.sum(num_false_pos) / num_total_negative
        em_overall_fpr = t_crit * np.std([n_folds * fp / num_total_negative for fp in num_false_pos]) / math.sqrt(n_folds)

        avg_energy_wastage_reduction = [f["stats_at_fpr"][fpr]["avg_energy_wastage_reduction"] for f in fold_data]
        mean_energy_wastage_reduction = np.mean(avg_energy_wastage_reduction)
        em_energy_wastage_reduction = t_crit * np.std(avg_energy_wastage_reduction) / math.sqrt(n_folds)

        avg_energy_wastage_reduction_pct = [f["stats_at_fpr"][fpr]["energy_wastage_reduction_pct"] for f in fold_data]
        mean_energy_wastage_reduction_pct = np.mean(avg_energy_wastage_reduction_pct)
        em_energy_wastage_reduction_pct = t_crit * np.std(avg_energy_wastage_reduction_pct) / math.sqrt(n_folds)

        batch_energy_wastage_reduction = [f["stats_at_fpr"][fpr]["energy_wastage_reduction"] for f in fold_data]
        mean_overall_energy_wastage_reduction_pct = 100 * np.sum(batch_energy_wastage_reduction) / total_energy_wastage
        em_overall_energy_wastage_reduction_pct = t_crit * np.std([(100 * n_folds * v / total_energy_wastage) for v in batch_energy_wastage_reduction]) / math.sqrt(n_folds)

        print(
            f"Batch FPR {fpr:.2f} ({round(fpr * num_negative)}/{num_negative}): "
            f"Overall FPR: {mean_overall_fpr:.4f} ± {em_overall_fpr:.4f}"
            f" | TPR: {mean_tpr:.4f} ± {std_tpr:.4f} ({round(mean_tpr * num_positive)}/{num_positive})"
            f" | Avg energy wastage reduction (batch) (mWh): {mean_energy_wastage_reduction:.4f} ± {em_energy_wastage_reduction:.4f}"
            f" | Reduction pct (batch): {mean_energy_wastage_reduction_pct:.4f} ± {em_energy_wastage_reduction_pct:.4f}"
            f" | Reduction pct (overall): {mean_overall_energy_wastage_reduction_pct:.4f} ± {em_overall_energy_wastage_reduction_pct:.4f}"
        )

        agg_fpr_stats.append({
            "mean_overall_fpr": mean_overall_fpr,
            "em_overall_fpr": em_overall_fpr,
            "mean_overall_energy_wastage_reduction_pct": mean_overall_energy_wastage_reduction_pct,
            "em_overall_energy_wastage_reduction_pct": em_overall_energy_wastage_reduction_pct,
        })

    return {
        "valid_idx": valid,
        "fold_data": fold_data,
        "mean_train_auc": mean_train_auc,
        "std_train_auc": std_train_auc,
        "mean_fold_auc": mean_fold_auc,
        "std_fold_auc": std_fold_auc,
        "agg_fpr_stats": agg_fpr_stats,
    }

In [ ]:
labels = [not c for c in is_correct]
train_res = []
for num_logprobs in [10]:
    for num_steps in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
        print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
        res = train_classifier(all_logprobs, all_tokens, all_data, labels, clf_step=num_steps, num_logprobs=num_logprobs)
        if res is not None:
            train_res.append(res)
        else:
            break
print([(res["mean_fold_auc"], res["std_fold_auc"]) for res in train_res])

In [ ]:
# Ablation for different number of logprobs
labels = [not c for c in is_correct]
for num_steps in [1, 2, 3, 4, 5]:
    train_res = []
    for num_logprobs in range(1, 21):
        print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
        res = train_classifier(all_logprobs, all_tokens, all_data, labels, clf_step=num_steps, num_logprobs=num_logprobs)
        if res is not None:
            train_res.append(res)
        else:
            break
    plt.plot(range(1, 21), [res["mean_fold_auc"] for res in train_res], marker="o")
    plt.grid(linestyle="--", color="lightgray")
    plt.xticks(range(1, 21))
    plt.show()
# print([(res["mean_fold_auc"], res["std_fold_auc"]) for res in train_res])

In [ ]:
plt.figure(figsize=(6, 4))

x = np.linspace(0, 100, 100)
plt.plot(x, x, color="silver", linestyle='--', alpha=0.5)

for i, res in enumerate(train_res):
    fpr_stats = res["agg_fpr_stats"]
    fprs = [100 * s["mean_overall_fpr"] for s in fpr_stats]
    # fprs = [fpr for fpr in fprs if fpr <= 20]
    # fprs_em = [s["em_overall_fpr"] for s in fpr_stats]
    energy_reductions = [s["mean_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]
    energy_reductions_em = [s["em_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]

    # plt.errorbar(fprs, energy_reductions, yerr=energy_reductions_em, ecolor="silver", label=f"Step {i + 1}", marker=".", capsize=3)
    plt.plot(fprs, energy_reductions, label=f"Step {i + 1}", marker=".", linewidth=3, alpha=0.8)

ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

# plt.xlabel("Task Performance Drop %")
# plt.ylabel("Energy Wastage Reduction %")
plt.xlim(-1, 26)
plt.ylim(-1, 26)
plt.yticks([0, 5, 10, 15, 20, 25])
plt.xticks([0, 5, 10, 15, 20, 25])
plt.tick_params(axis="both", labelsize=20)
# plt.legend(fontsize=20, labelspacing=0.1, borderpad=0.0, ncols=3, frameon=False)
plt.savefig(f"../figures/energy_vs_fpr_{dataset}_qwen3_{size}b.pdf", bbox_inches="tight", pad_inches=0.0)

# handles, labels = ax.get_legend_handles_labels()
# fig_leg = plt.figure(figsize=(4, 0.5))
# ax_leg = fig_leg.add_subplot(111)
# ax_leg.axis("off")  # hide axes
# legend = ax_leg.legend(handles, labels, loc="center", ncol=5, frameon=False, borderpad=0.0, columnspacing=1)
# fig_leg.savefig("../figures/energy_vs_fpr_legend.pdf", bbox_inches="tight", pad_inches=0.0)
plt.show()

In [ ]:
auc_data = {
    "30B-A3B, FRAMES": [(np.float64(0.5943658536585367), np.float64(0.06484981964257111)), (np.float64(0.6233153357961542), np.float64(0.040178834461863246)), (np.float64(0.6422304465023261), np.float64(0.03993045377447523)), (np.float64(0.6874033339061694), np.float64(0.07630765301848012)), (np.float64(0.7649839743589744), np.float64(0.0868050474297141)), (np.float64(0.6639520202020202), np.float64(0.19120708281339466)), (np.float64(0.6416666666666667), np.float64(0.21219749710535646)), (np.float64(0.45555555555555555), np.float64(0.209054308024742)), (np.float64(0.55), np.float64(0.20652617564001371)), (np.float64(0.6066666666666667), np.float64(0.20044395171163878))],
    "1.7B, FRAMES": [(np.float64(0.499467532787087), np.float64(0.0324059626464355)), (np.float64(0.48737901664730937), np.float64(0.0523512569916332)), (np.float64(0.40247899159663864), np.float64(0.10486669402288064)), (np.float64(0.4838981331168831), np.float64(0.11064472883577958)), (np.float64(0.5015841013824884), np.float64(0.15066689943078224)), (np.float64(0.47006048387096777), np.float64(0.10527016995692705)), (np.float64(0.4413306451612904), np.float64(0.09828628926756805)), (np.float64(0.39400921658986177), np.float64(0.08794481443661015)), (np.float64(0.44715821812596007), np.float64(0.07738704126558923)), (np.float64(0.45714285714285713), np.float64(0.0837341264791438))],
    "30B-A3B, SimpleQA": [(np.float64(0.5511485081687696), np.float64(0.010229883581904978)), (np.float64(0.701668885166532), np.float64(0.013725503707679877)), (np.float64(0.7177959003846397), np.float64(0.03292154254679271)), (np.float64(0.6608021199932964), np.float64(0.08630561597775527)), (np.float64(0.5440128098022835), np.float64(0.07184337344809168)), (np.float64(0.34412698412698417), np.float64(0.21071905354364387)), (np.float64(0.43), np.float64(0.14))],
    "1.7B, SimpleQA": [(np.float64(0.5572214130277093), np.float64(0.014334837271561257)), (np.float64(0.4996428571428571), np.float64(0.11992496814416483)), (np.float64(0.5442810457516339), np.float64(0.08657995482236643)), (np.float64(0.5325641025641026), np.float64(0.12569752784983806)), (np.float64(0.471969696969697), np.float64(0.10764695680537477)), (np.float64(0.6236363636363638), np.float64(0.13770251802735628))],
}

model_colors = {
    "30B-A3B": "#56B4E9",
    "1.7B": "#CC79A7",
}

dataset_markers = {
    "FRAMES": "o",
    "SimpleQA": "s",
}

plt.figure(figsize=(6,4))
fontsize=10

offset = -0.15
offset_delta = 0.075

for label, aucs in auc_data.items():
    model, ds = label.split(", ")
    means = [e[0] for e in aucs]
    stds = [e[1] for e in aucs]

    n = 5
    se = np.array(stds) / math.sqrt(n)

    # t critical value
    t_crit = stats.t.ppf((1 + 0.95) / 2, df=n-1)

    # margin of error
    margin_error = t_crit * se

    plt.errorbar(
        np.arange(1, len(means) + 1) + offset, means, yerr=margin_error, label=label,
        ecolor="silver", color=model_colors[model], marker=dataset_markers[ds], capsize=3,
    )
    offset += offset_delta

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

plt.xticks(np.arange(1, 11))
# plt.xlabel("Step", fontsize=fontsize)

plt.yticks([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
# plt.ylabel("ROC-AUC", fontsize=fontsize)
plt.tick_params(axis="both", labelsize=fontsize)

custom_legend = []
for label, color in model_colors.items():
    custom_legend.append(Line2D([0], [0], color=color, lw=2, label=label))
for label, marker in dataset_markers.items():
    custom_legend.append(Line2D([0], [0], color="black", marker=marker, lw=2, label=label))
plt.legend(handles=custom_legend, fontsize=fontsize, frameon=False, labelspacing=0.1, borderpad=0.0, loc="lower right")
# plt.legend(handles=custom_legend, fontsize=fontsize, labelspacing=0.1, handlelength=1.5, borderpad=0.1, frameon=False, bbox_to_anchor=(1.025, 1.025), loc="upper right")
# plt.legend(fontsize=16, labelspacing=0.1, frameon=False, bbox_to_anchor=(0.5, -0.8), loc="lower center")

plt.savefig("../figures/auc_qa.pdf", bbox_inches="tight", pad_inches=0.0)
plt.show()


In [ ]:
# Display a correlation heatmap of the features and label
labels = [c for c in is_correct]
X, y, _ = extract_features(all_logprobs, all_tokens, labels, clf_step=3)
plt.figure(figsize=(12, 8))
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
df['label'] = y
sns.heatmap(df.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
def eval_baseline(
    logprobs_data,
    all_data,
    labels,
    clf_step,
):
    num_total_negative = len(labels) - sum(labels)
    total_energy_wastage = all_data[~all_data["agent_output_is_correct"]]["total_energy"].sum()
    
    min_step_logprobs = []
    mean_step_logprobs = []
    y = []
    valid = []
    for i, logprobs in enumerate(logprobs_data):
        if len(logprobs) <= clf_step:
            continue
        step_logprobs = logprobs[clf_step - 1]
        min_step_logprobs.append(min(step_logprobs))
        mean_step_logprobs.append(np.mean(step_logprobs))
        y.append(int(labels[i]))
        valid.append(i)

    y = np.array(y)
    valid_data = all_data.iloc[valid]

    stat_dicts = {
        "min_logprob": stats_at_fpr(y, min_step_logprobs, fpr_targets, clf_step, valid_data),
        "mean_logprob": stats_at_fpr(y, mean_step_logprobs, fpr_targets, clf_step, valid_data),
        "random": stats_at_fpr(y, np.random.random(len(y)), fpr_targets, clf_step, valid_data),
    }

    agg_fpr_stats = {}
    for label, stat_dict in stat_dicts.items():
        agg_fpr_stats[label] = []
        for fpr in fpr_targets:
            fpr_stats = stat_dict[fpr]
            num_false_pos = fpr_stats["num_false_pos"]
            mean_overall_fpr = np.sum(num_false_pos) / num_total_negative

            batch_energy_wastage_reduction = fpr_stats["energy_wastage_reduction"]
            mean_overall_energy_wastage_reduction_pct = 100 * batch_energy_wastage_reduction / total_energy_wastage

            agg_fpr_stats[label].append({
                "mean_overall_fpr": mean_overall_fpr,
                "mean_overall_energy_wastage_reduction_pct": mean_overall_energy_wastage_reduction_pct,
            })

    return agg_fpr_stats

labels = [not c for c in is_correct]
baseline_res = []
for num_steps in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    res = eval_baseline(all_logprobs, all_data, labels, clf_step=num_steps)
    baseline_res.append(res)


In [ ]:
plt.figure(figsize=(6, 4))

# Default
x = np.linspace(0, 100, 100)
plt.plot(x, x, color="silver", linestyle='--', alpha=0.5)

linestyle_dict = {
    "random": "--",
    "min_logprob": "dotted",
    "mean_logprob": "-.",
}

colors = {}

method_n = 0
for method, linestyle in linestyle_dict.items():
    method_n += 1
    for i, fpr_stat_methods in enumerate(baseline_res):
        fpr_stats = fpr_stat_methods[method]
        fprs = [100 * s["mean_overall_fpr"] for s in fpr_stats]
        energy_reductions = [s["mean_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]
        if method_n == 1:
            line, = plt.plot(fprs, energy_reductions, linestyle=linestyle_dict[method], linewidth=3, alpha=0.7)
            colors[i] = line.get_color()
        else:
            plt.plot(fprs, energy_reductions, linestyle=linestyle_dict[method], linewidth=3, alpha=0.7, color=colors[i])

ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

# plt.xlabel("Task Performance Drop %")
# plt.ylabel("Energy Wastage Reduction %")
plt.xlim(-1, 26)
plt.ylim(-1, 26)
plt.yticks([0, 5, 10, 15, 20, 25])
plt.xticks([0, 5, 10, 15, 20, 25])
plt.tick_params(axis="both", labelsize=20)

# Create custom legend handles for linestyles (black color)
# linestyle_handles = [Line2D([0], [0], color='black', lw=2, linestyle=ls) 
#                      for ls in linestyle_dict.values()]
# linestyle_labels = list(linestyle_dict.keys())

# # Combine handles and labels
# handles = []
# labels = []

# plt.legend(linestyle_handles, linestyle_labels, fontsize=20, loc="upper left", labelspacing=0.1, borderpad=0.0, frameon=False, columnspacing=1)
plt.savefig(f"../figures/energy_vs_fpr_{dataset}_qwen3_{size}b_baselines.pdf", bbox_inches="tight", pad_inches=0.0)
plt.show()

In [ ]:
# Generalizability across datasets

fpr_targets = [0.05 * i for i in range(21)]

def train_classifier_cross_dataset(
    train_data,
    val_data,
    clf_step,
    n_folds=5,
    num_logprobs=10,
):
    train_logprobs_data, train_tokens_data, train_is_correct, _, train_all_data = train_data
    val_logprobs_data, val_tokens_data, val_is_correct, _, val_all_data= val_data

    train_labels = [not c for c in train_is_correct]
    val_labels = [not c for c in val_is_correct]

    X_train, y_train, valid_train = extract_features(
        train_logprobs_data,
        train_tokens_data,
        train_labels,
        clf_step,
        num_logprobs=num_logprobs,
    )
    X_val, y_val, valid_val = extract_features(
        val_logprobs_data,
        val_tokens_data,
        val_labels,
        clf_step,
        num_logprobs=num_logprobs,
    )

    num_total = len(y_train)
    num_positive = y_train.sum()
    num_negative = num_total - num_positive

    if min(num_positive, num_negative) <= n_folds:
        return
    
    valid_val_data = val_all_data.iloc[valid_val]
    
    print(f"X dimension: {X_train.shape}")
    print(f"y positive: {y_train.mean():.3f} ({num_positive}/{num_total})")

    base_estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    param_dist = {
        "max_depth": stats.randint(3, 8),
        "learning_rate": stats.loguniform(0.005, 0.2),
        "min_child_weight": stats.randint(1, 8),
        "subsample": stats.uniform(0.7, 0.3),
        "colsample_bytree": stats.uniform(0.7, 0.3),
        "gamma": stats.uniform(0, 5),

    }

    search = HalvingRandomSearchCV(
        estimator=base_estimator,
        param_distributions=param_dist,
        scoring="roc_auc",
        n_jobs=-1,
        cv=n_folds,
        factor=3,
        resource="n_estimators",
        max_resources=300,
        min_resources=10,
        random_state=42,
        verbose=0,
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # print("Best params:", search.best_params_)

    train_proba = best_model.predict_proba(X_train)[:, 1]
    val_proba = best_model.predict_proba(X_val)[:, 1]

    train_auc_score = roc_auc_score(y_train, train_proba)
    val_auc_score = roc_auc_score(y_val, val_proba)
    
    stats_at_fpr_dict = stats_at_fpr(y_val, val_proba, fpr_targets, clf_step, valid_val_data)

    # -------------------------
    # Summary
    # -------------------------

    print("\n" + "=" * 50)
    print("FINAL NESTED CV RESULTS")
    print("=" * 50)

    num_val_total_negative = len(val_labels) - sum(val_labels)

    print(f"Train ROC-AUC: {train_auc_score:.4f}")
    print(f"Test ROC-AUC: {val_auc_score:.4f} ")

    print("TPR and Energy Wastage @ FPR summary:")
    agg_fpr_stats = []
    for fpr in fpr_targets:
        fpr_data = stats_at_fpr_dict[fpr]
        tpr = fpr_data["tpr"]
        num_false_pos = fpr_data["num_false_pos"]
        mean_overall_fpr = num_false_pos / num_val_total_negative

        avg_energy_wastage_reduction = fpr_data["avg_energy_wastage_reduction"]
        avg_energy_wastage_reduction_pct = fpr_data["energy_wastage_reduction_pct"]

        print(
            f"Batch FPR {fpr:.2f} ({round(fpr * num_negative)}/{num_negative}): "
            f"Overall FPR: {mean_overall_fpr:.4f}"
            f" | TPR: {tpr:.4f} ({round(tpr * num_positive)}/{num_positive})"
            f" | Avg energy wastage reduction (batch) (mWh): {avg_energy_wastage_reduction:.4f}"
            f" | Reduction pct (overall): {avg_energy_wastage_reduction_pct:.4f}"
        )

        agg_fpr_stats.append({
            "mean_overall_fpr": mean_overall_fpr,
            "mean_overall_energy_wastage_reduction_pct": avg_energy_wastage_reduction_pct,
        })

    return {
        "valid_idx": valid_val,
        "mean_train_auc": train_auc_score,
        "mean_fold_auc": val_auc_score,
        "agg_fpr_stats": agg_fpr_stats,
    }

In [ ]:
source_data = load_data("frames", "30")
target_data = load_data("frames", "1.7")

In [ ]:
train_res = []
for num_logprobs in [10]:
    for num_steps in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
        print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
        res = train_classifier_cross_dataset(source_data, target_data, clf_step=num_steps, num_logprobs=num_logprobs)
        if res is not None:
            train_res.append(res)
        else:
            break
print([(res["mean_train_auc"], res["mean_fold_auc"]) for res in train_res])

In [ ]:
# Feature importance

import shap

def extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=10):
    X_features = []
    y = []
    valid = []

    for i, (logprobs, tokens) in enumerate(zip(logprobs_data, tokens_data)):
        if len(logprobs) <= clf_step:
            continue
        feature_vector = []

        probs = [np.exp(lp) for lp in logprobs[:clf_step]]
        min_logprobs = [v for p in probs for v in np.sort(p)[:num_logprobs]]
        feature_vector.extend(min_logprobs)

        # Token features
        num_tokens = [len(lp) for lp in logprobs[:clf_step]]
        feature_vector.extend(num_tokens)

        num_thought_tokens = [(step_tokens.index("<code") if "<code" in step_tokens else len(step_tokens)) for step_tokens in tokens[:clf_step]]
        feature_vector.extend(num_thought_tokens)

        # Length of longest common substring between current step and previous step
        for j in range(max(clf_step - 1, 1), clf_step):
            cur_gen = "".join(tokens[j])
            prev_gen = "".join(tokens[j - 1])
            feature_vector.append(len(lcs(cur_gen, prev_gen)) / len(cur_gen))

        X_features.append(feature_vector)
        y.append(int(labels[i]))
        valid.append(i)

    if all(y) or not any(y):
        print("Not enough valid labels")
        raise Exception

    max_num_feature = max(len(f) for f in X_features)
    correct_record_idx = [i for i in range(len(X_features)) if len(X_features[i]) == max_num_feature]
    valid = [valid_idx for i, valid_idx in enumerate(valid) if len(X_features[i]) == max_num_feature]

    X_features = [f for f in X_features if len(f) == max_num_feature]
    X = np.array(X_features)
    y = np.array(y)[correct_record_idx]

    names = []
    for step in range(clf_step):
        for i in range(num_logprobs):
            names.append(f"step_{step}_top_{i + 1}_min_logprobs")
        names.extend([f"step_{step}_num_tokens", f"step_{step}_num_thought_tokens"])
    if clf_step > 1:
        names.append(f"token_overlap")
    
    return X, y, valid, names

def feature_importance(
    train_data,
    clf_step,
    n_folds=5,
    num_logprobs=10,
):
    train_logprobs_data, train_tokens_data, train_is_correct, _, train_all_data, _ = train_data

    train_labels = [not c for c in train_is_correct]

    X_train, y_train, valid_train, feature_names = extract_features(
        train_logprobs_data,
        train_tokens_data,
        train_labels,
        clf_step,
        num_logprobs=num_logprobs,
    )

    num_total = len(y_train)
    num_positive = y_train.sum()
    num_negative = num_total - num_positive

    if min(num_positive, num_negative) <= n_folds:
        return
    
    print(f"X dimension: {X_train.shape}")
    print(f"y positive: {y_train.mean():.3f} ({num_positive}/{num_total})")

    base_estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    param_dist = {
        "max_depth": stats.randint(3, 8),
        "learning_rate": stats.loguniform(0.005, 0.2),
        "min_child_weight": stats.randint(1, 8),
        "subsample": stats.uniform(0.7, 0.3),
        "colsample_bytree": stats.uniform(0.7, 0.3),
        "gamma": stats.uniform(0, 5),

    }

    search = HalvingRandomSearchCV(
        estimator=base_estimator,
        param_distributions=param_dist,
        scoring="roc_auc",
        n_jobs=-1,
        cv=n_folds,
        factor=3,
        resource="n_estimators",
        max_resources=300,
        min_resources=10,
        random_state=42,
        verbose=0,
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    # importances = best_model.feature_importances_

    # # Pair with feature names
    # for name, score in zip(feature_names, importances):
    #     print(f"{name}: {score}")

    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_train)
    importance = np.abs(shap_values).mean(axis=0)

    for name, val in sorted(zip(feature_names, importance), key=lambda x: -x[1]):
        print(name, val)

In [ ]:
data = load_data("frames", "30")
for step in range(10):
    feature_importance(load_data("frames", "30"), clf_step=step+1)